# Лабораторная работа №8: Решение задачи коммивояжера с помощью генетического алгоритма

**Цель работы:** реализовать генетический алгоритм для решения задачи коммивояжера (TSP) с использованием перестановочного кодирования и циклического кроссинговера.

**Задачи:**
1. Загрузить набор данных eil76.tsp (76 городов) из библиотеки TSPLIB.
2. Реализовать перестановочное представление хромосом (маршрут = перестановка индексов городов).
3. Реализовать циклический кроссинговер (CX) — вариант 3.
4. Реализовать мутацию обменом (swap mutation), турнирную селекцию и элитистскую стратегию отбора.
5. Визуализировать карту городов, процесс сходимости, лучший маршрут и анимацию улучшения.
6. Исследовать влияние гиперпараметров на качество решения.
7. Сравнить полученный результат с известным оптимальным решением (538).

**Вариант 3:** Циклический кроссинговер (CX), тестовый файл eil76.tsp.

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

OUTPUTS = Path("report/outputs")
OUTPUTS.mkdir(exist_ok=True)

np.random.seed(42)

## 1. Загрузка и разбор данных eil76.tsp

Файл eil76.tsp содержит 76 городов в формате TSPLIB с евклидовыми координатами.
Известное оптимальное решение: **538**.

In [ ]:
OPTIMAL_TOUR_LENGTH = 538


def parse_tsp(filepath: str) -> tuple[np.ndarray, list[str]]:
    """Разбор файла TSPLIB. Возвращает координаты и метки городов."""
    coords = []
    names = []
    reading = False
    with open(filepath) as f:
        for line in f:
            line = line.strip()
            if line == "NODE_COORD_SECTION":
                reading = True
                continue
            if line in ("EOF", ""):
                if reading:
                    break
                continue
            if reading:
                parts = line.split()
                names.append(parts[0])
                coords.append((float(parts[1]), float(parts[2])))
    return np.array(coords), names


coords, city_names = parse_tsp("data/eil76.tsp")
N_CITIES = len(coords)
print(f"Загружено городов: {N_CITIES}")
print(f"Первые 5 координат:\n{coords[:5]}")

## 2. Матрица расстояний и функция оценки маршрута

In [ ]:
def compute_distance_matrix(coords: np.ndarray) -> np.ndarray:
    """Матрица евклидовых расстояний между всеми парами городов."""
    diff = coords[:, np.newaxis, :] - coords[np.newaxis, :, :]
    return np.sqrt((diff ** 2).sum(axis=2))


dist_matrix = compute_distance_matrix(coords)
print(f"Размер матрицы расстояний: {dist_matrix.shape}")
print(f"Расстояние между городами 0 и 1: {dist_matrix[0, 1]:.2f}")

In [ ]:
def tour_length(tour: np.ndarray, dist: np.ndarray) -> float:
    """Длина замкнутого маршрута."""
    total = 0.0
    for i in range(len(tour)):
        total += dist[tour[i], tour[(i + 1) % len(tour)]]
    return total


random_tour = np.random.permutation(N_CITIES)
print(f"Длина случайного маршрута: {tour_length(random_tour, dist_matrix):.2f}")
print(f"Оптимальная длина: {OPTIMAL_TOUR_LENGTH}")

## 3. Визуализация карты городов

In [ ]:
fig_cities = go.Figure()
fig_cities.add_trace(go.Scatter(
    x=coords[:, 0], y=coords[:, 1],
    mode="markers+text",
    marker=dict(size=8, color="#636EFA"),
    text=city_names,
    textposition="top center",
    textfont=dict(size=7),
    name="Города",
))
fig_cities.update_layout(
    title="Карта 76 городов (eil76)",
    xaxis_title="X", yaxis_title="Y",
    width=800, height=700,
    template="plotly_white",
    showlegend=False,
)
fig_cities.write_html(OUTPUTS / "01_cities_map.html")
fig_cities.show()

## 4. Генетические операторы

### 4.1 Циклический кроссинговер (CX)

Циклический кроссинговер сохраняет абсолютные позиции элементов из родителей:
1. Потомок начинает формирование с первого гена родителя P1.
2. Смотрим, какой элемент стоит на той же позиции в P2 — ищем его позицию в P1.
3. Берём элемент P1 с этой позиции, повторяем до замыкания цикла.
4. Оставшиеся позиции заполняются из P2.

### 4.2 Мутация обменом (Swap Mutation)

Случайным образом выбираются две позиции в хромосоме и обмениваются местами.

### 4.3 Турнирная селекция

Из популяции случайно выбирается $k$ особей, лучшая из них становится родителем.

In [ ]:
def cyclic_crossover(p1: np.ndarray, p2: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """
    Циклический кроссинговер (CX).
    Возвращает двух потомков, каждый из которых наследует
    позиции из одного родителя по циклам.
    """
    n = len(p1)
    child1 = np.full(n, -1, dtype=int)
    child2 = np.full(n, -1, dtype=int)

    p2_pos = {val: idx for idx, val in enumerate(p2)}
    p1_pos = {val: idx for idx, val in enumerate(p1)}

    cycle_num = 0
    visited = np.zeros(n, dtype=bool)

    for start in range(n):
        if visited[start]:
            continue
        idx = start
        while not visited[idx]:
            visited[idx] = True
            if cycle_num % 2 == 0:
                child1[idx] = p1[idx]
                child2[idx] = p2[idx]
            else:
                child1[idx] = p2[idx]
                child2[idx] = p1[idx]
            idx = p1_pos[p2[idx]]
        cycle_num += 1

    return child1, child2


def swap_mutation(tour: np.ndarray) -> np.ndarray:
    """Мутация обменом: обмен двух случайных позиций."""
    mutant = tour.copy()
    i, j = np.random.choice(len(tour), size=2, replace=False)
    mutant[i], mutant[j] = mutant[j], mutant[i]
    return mutant


def tournament_selection(
    population: np.ndarray, fitnesses: np.ndarray, k: int = 3
) -> np.ndarray:
    """Турнирная селекция: выбираем лучшего из k случайных."""
    indices = np.random.choice(len(population), size=k, replace=False)
    best = indices[np.argmin(fitnesses[indices])]
    return population[best].copy()

In [ ]:
test_p1 = np.array([0, 1, 2, 3, 4, 5, 6, 7])
test_p2 = np.array([7, 6, 5, 4, 3, 2, 1, 0])
c1, c2 = cyclic_crossover(test_p1, test_p2)
print("Тест CX:")
print(f"  P1: {test_p1}")
print(f"  P2: {test_p2}")
print(f"  C1: {c1}  (все элементы уникальны: {len(set(c1)) == len(c1)})")
print(f"  C2: {c2}  (все элементы уникальны: {len(set(c2)) == len(c2)})")

## 5. Основной цикл генетического алгоритма

In [ ]:
POP_SIZE = 150
N_GENERATIONS = 500
P_CROSSOVER = 0.9
P_MUTATION = 0.2
TOURNAMENT_K = 3
ELITE_SIZE = 2


def run_tsp_ga(
    dist: np.ndarray,
    n_cities: int,
    pop_size: int = POP_SIZE,
    n_gen: int = N_GENERATIONS,
    p_cx: float = P_CROSSOVER,
    p_mut: float = P_MUTATION,
    k: int = TOURNAMENT_K,
    elite: int = ELITE_SIZE,
    seed: int | None = None,
    save_every: int = 25,
) -> dict:
    """
    Генетический алгоритм для задачи коммивояжера.
    Возвращает словарь с историей и лучшим решением.
    """
    if seed is not None:
        np.random.seed(seed)

    population = np.array([np.random.permutation(n_cities) for _ in range(pop_size)])
    fitnesses = np.array([tour_length(ind, dist) for ind in population])

    best_history = []
    avg_history = []
    tour_snapshots = {}

    global_best_idx = np.argmin(fitnesses)
    global_best_tour = population[global_best_idx].copy()
    global_best_fit = fitnesses[global_best_idx]

    for gen in range(n_gen):
        best_idx = np.argmin(fitnesses)
        best_fit = fitnesses[best_idx]
        avg_fit = fitnesses.mean()
        best_history.append(best_fit)
        avg_history.append(avg_fit)

        if best_fit < global_best_fit:
            global_best_fit = best_fit
            global_best_tour = population[best_idx].copy()

        if gen % save_every == 0 or gen == n_gen - 1:
            tour_snapshots[gen] = population[best_idx].copy()

        sorted_idx = np.argsort(fitnesses)
        elites = population[sorted_idx[:elite]].copy()

        new_population = list(elites)

        while len(new_population) < pop_size:
            parent1 = tournament_selection(population, fitnesses, k)
            parent2 = tournament_selection(population, fitnesses, k)

            if np.random.random() < p_cx:
                child1, child2 = cyclic_crossover(parent1, parent2)
            else:
                child1, child2 = parent1.copy(), parent2.copy()

            if np.random.random() < p_mut:
                child1 = swap_mutation(child1)
            if np.random.random() < p_mut:
                child2 = swap_mutation(child2)

            new_population.append(child1)
            if len(new_population) < pop_size:
                new_population.append(child2)

        population = np.array(new_population[:pop_size])
        fitnesses = np.array([tour_length(ind, dist) for ind in population])

    final_best_idx = np.argmin(fitnesses)
    if fitnesses[final_best_idx] < global_best_fit:
        global_best_fit = fitnesses[final_best_idx]
        global_best_tour = population[final_best_idx].copy()

    best_history.append(global_best_fit)
    avg_history.append(fitnesses.mean())

    return {
        "best_tour": global_best_tour,
        "best_length": global_best_fit,
        "best_history": best_history,
        "avg_history": avg_history,
        "snapshots": tour_snapshots,
    }

In [ ]:
result = run_tsp_ga(dist_matrix, N_CITIES, seed=42)

print(f"Лучший найденный маршрут: {result['best_length']:.2f}")
print(f"Известный оптимум: {OPTIMAL_TOUR_LENGTH}")
print(f"Отклонение от оптимума: {(result['best_length'] - OPTIMAL_TOUR_LENGTH) / OPTIMAL_TOUR_LENGTH * 100:.2f}%")

## 6. Визуализация сходимости

In [ ]:
generations = list(range(len(result["best_history"])))

fig_conv = make_subplots(
    rows=1, cols=1,
    subplot_titles=["Сходимость генетического алгоритма"],
)

fig_conv.add_trace(go.Scatter(
    x=generations, y=result["best_history"],
    mode="lines", name="Лучший маршрут",
    line=dict(color="#636EFA", width=2),
))
fig_conv.add_trace(go.Scatter(
    x=generations, y=result["avg_history"],
    mode="lines", name="Средняя длина",
    line=dict(color="#EF553B", width=1.5, dash="dash"),
))
fig_conv.add_hline(
    y=OPTIMAL_TOUR_LENGTH, line_dash="dot",
    line_color="green", annotation_text=f"Оптимум = {OPTIMAL_TOUR_LENGTH}",
    annotation_position="top left",
)

fig_conv.update_layout(
    xaxis_title="Поколение",
    yaxis_title="Длина маршрута",
    width=900, height=500,
    template="plotly_white",
    legend=dict(x=0.7, y=0.95),
)
fig_conv.write_html(OUTPUTS / "02_convergence.html")
fig_conv.show()

## 7. Лучший найденный маршрут

In [ ]:
def plot_tour(tour: np.ndarray, coords: np.ndarray, title: str) -> go.Figure:
    """Визуализация маршрута на карте городов."""
    ordered = coords[tour]
    loop = np.vstack([ordered, ordered[0:1]])

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=loop[:, 0], y=loop[:, 1],
        mode="lines",
        line=dict(color="#636EFA", width=1.5),
        name="Маршрут",
    ))
    fig.add_trace(go.Scatter(
        x=coords[:, 0], y=coords[:, 1],
        mode="markers+text",
        marker=dict(size=7, color="#EF553B"),
        text=[str(i + 1) for i in range(len(coords))],
        textposition="top center",
        textfont=dict(size=6),
        name="Города",
    ))
    fig.update_layout(
        title=title,
        xaxis_title="X", yaxis_title="Y",
        width=800, height=700,
        template="plotly_white",
        showlegend=True,
    )
    return fig


fig_best = plot_tour(
    result["best_tour"], coords,
    f"Лучший маршрут (длина = {result['best_length']:.2f}, оптимум = {OPTIMAL_TOUR_LENGTH})",
)
fig_best.write_html(OUTPUTS / "03_best_tour.html")
fig_best.show()

## 8. Анимация улучшения маршрута по поколениям

In [ ]:
snapshots = result["snapshots"]
snap_gens = sorted(snapshots.keys())

frames = []
for gen in snap_gens:
    tour = snapshots[gen]
    ordered = coords[tour]
    loop = np.vstack([ordered, ordered[0:1]])
    length = tour_length(tour, dist_matrix)
    frames.append(go.Frame(
        data=[
            go.Scatter(
                x=loop[:, 0], y=loop[:, 1],
                mode="lines",
                line=dict(color="#636EFA", width=1.5),
            ),
            go.Scatter(
                x=coords[:, 0], y=coords[:, 1],
                mode="markers",
                marker=dict(size=7, color="#EF553B"),
            ),
        ],
        name=str(gen),
        layout=go.Layout(
            title=f"Поколение {gen} — длина маршрута: {length:.2f}"
        ),
    ))

first_tour = snapshots[snap_gens[0]]
first_ordered = coords[first_tour]
first_loop = np.vstack([first_ordered, first_ordered[0:1]])
first_len = tour_length(first_tour, dist_matrix)

fig_anim = go.Figure(
    data=[
        go.Scatter(
            x=first_loop[:, 0], y=first_loop[:, 1],
            mode="lines",
            line=dict(color="#636EFA", width=1.5),
            name="Маршрут",
        ),
        go.Scatter(
            x=coords[:, 0], y=coords[:, 1],
            mode="markers",
            marker=dict(size=7, color="#EF553B"),
            name="Города",
        ),
    ],
    frames=frames,
    layout=go.Layout(
        title=f"Поколение {snap_gens[0]} — длина маршрута: {first_len:.2f}",
        xaxis_title="X", yaxis_title="Y",
        width=800, height=700,
        template="plotly_white",
        updatemenus=[
            dict(
                type="buttons",
                showactive=False,
                y=1.12, x=0.5, xanchor="center",
                buttons=[
                    dict(label="▶ Воспроизвести",
                         method="animate",
                         args=[None, {
                             "frame": {"duration": 500, "redraw": True},
                             "fromcurrent": True,
                             "transition": {"duration": 200},
                         }]),
                    dict(label="⏸ Пауза",
                         method="animate",
                         args=[[None], {
                             "frame": {"duration": 0, "redraw": False},
                             "mode": "immediate",
                             "transition": {"duration": 0},
                         }]),
                ],
            )
        ],
        sliders=[dict(
            active=0,
            currentvalue={"prefix": "Поколение: "},
            steps=[
                dict(args=[[str(gen)], {"frame": {"duration": 300, "redraw": True},
                                        "mode": "immediate"}],
                     label=str(gen), method="animate")
                for gen in snap_gens
            ],
        )],
    ),
)

fig_anim.write_html(OUTPUTS / "04_tour_animation.html")
fig_anim.show()

## 9. Анализ влияния параметров

Исследуем зависимость качества решения от:
- размера популяции,
- вероятности кроссинговера,
- вероятности мутации,
- числа поколений.

In [ ]:
def run_experiment(**kwargs) -> float:
    """Запуск ГА с заданными параметрами, возврат лучшей длины."""
    res = run_tsp_ga(dist_matrix, N_CITIES, **kwargs)
    return res["best_length"]


param_configs = {
    "Размер популяции": {
        "param": "pop_size",
        "values": [50, 100, 150, 200, 300],
        "defaults": {"n_gen": 300, "p_cx": 0.9, "p_mut": 0.2, "seed": 42},
    },
    "P кроссинговера": {
        "param": "p_cx",
        "values": [0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
        "defaults": {"pop_size": 150, "n_gen": 300, "p_mut": 0.2, "seed": 42},
    },
    "P мутации": {
        "param": "p_mut",
        "values": [0.05, 0.1, 0.2, 0.3, 0.5],
        "defaults": {"pop_size": 150, "n_gen": 300, "p_cx": 0.9, "seed": 42},
    },
    "Число поколений": {
        "param": "n_gen",
        "values": [100, 200, 300, 500, 800],
        "defaults": {"pop_size": 150, "p_cx": 0.9, "p_mut": 0.2, "seed": 42},
    },
}

param_results = {}
for name, cfg in param_configs.items():
    results_for_param = []
    for val in cfg["values"]:
        kwargs = {**cfg["defaults"], cfg["param"]: val}
        best = run_experiment(**kwargs)
        results_for_param.append(best)
        print(f"  {name} = {val}: лучшая длина = {best:.2f}")
    param_results[name] = (cfg["values"], results_for_param)
    print()

In [ ]:
fig_params = make_subplots(
    rows=2, cols=2,
    subplot_titles=list(param_results.keys()),
    horizontal_spacing=0.12,
    vertical_spacing=0.14,
)

colors = ["#636EFA", "#EF553B", "#00CC96", "#AB63FA"]
positions = [(1, 1), (1, 2), (2, 1), (2, 2)]

for i, (name, (vals, fits)) in enumerate(param_results.items()):
    r, c = positions[i]
    fig_params.add_trace(
        go.Scatter(
            x=[str(v) for v in vals], y=fits,
            mode="lines+markers",
            marker=dict(size=8, color=colors[i]),
            line=dict(color=colors[i], width=2),
            name=name,
            showlegend=False,
        ),
        row=r, col=c,
    )
    fig_params.add_hline(
        y=OPTIMAL_TOUR_LENGTH, line_dash="dot", line_color="gray",
        row=r, col=c,
    )
    fig_params.update_yaxes(title_text="Длина маршрута", row=r, col=c)

fig_params.update_layout(
    title="Анализ чувствительности параметров",
    width=1000, height=700,
    template="plotly_white",
)
fig_params.write_html(OUTPUTS / "05_param_analysis.html")
fig_params.show()

## 10. Итоговая таблица результатов

In [ ]:
gap = (result["best_length"] - OPTIMAL_TOUR_LENGTH) / OPTIMAL_TOUR_LENGTH * 100

table_data = {
    "Параметр": [
        "Задача",
        "Число городов",
        "Кроссинговер",
        "Мутация",
        "Селекция",
        "Редукция",
        "Размер популяции",
        "Число поколений",
        "P кроссинговера",
        "P мутации",
        "Размер турнира",
        "Элитизм",
        "Известный оптимум",
        "Найденное решение",
        "Отклонение от оптимума",
    ],
    "Значение": [
        "eil76 (Christofides/Eilon)",
        str(N_CITIES),
        "Циклический (CX)",
        "Обмен (Swap)",
        f"Турнирная (k={TOURNAMENT_K})",
        f"Элитистская ({ELITE_SIZE} лучших)",
        str(POP_SIZE),
        str(N_GENERATIONS),
        str(P_CROSSOVER),
        str(P_MUTATION),
        str(TOURNAMENT_K),
        str(ELITE_SIZE),
        str(OPTIMAL_TOUR_LENGTH),
        f"{result['best_length']:.2f}",
        f"{gap:.2f}%",
    ],
}

fig_table = go.Figure(data=[go.Table(
    header=dict(
        values=["<b>Параметр</b>", "<b>Значение</b>"],
        fill_color="#636EFA",
        font=dict(color="white", size=13),
        align="left",
        height=35,
    ),
    cells=dict(
        values=[table_data["Параметр"], table_data["Значение"]],
        fill_color=[["#f0f2ff" if i % 2 == 0 else "white"
                      for i in range(len(table_data["Параметр"]))]]*2,
        font=dict(size=12),
        align="left",
        height=28,
    ),
)])

fig_table.update_layout(
    title="Сводная таблица результатов",
    width=700, height=580,
    margin=dict(l=20, r=20, t=50, b=20),
)
fig_table.write_html(OUTPUTS / "06_results_table.html")
fig_table.show()

## Выводы

1. Реализован генетический алгоритм для решения задачи коммивояжера на наборе данных eil76 (76 городов).
2. Использовано перестановочное представление хромосом — каждая особь представляет собой перестановку индексов городов.
3. Реализован **циклический кроссинговер (CX)**, который сохраняет абсолютные позиции элементов из родителей и гарантирует допустимость потомков (без повторов городов).
4. Мутация обменом, турнирная селекция (k=3) и элитистская стратегия обеспечивают баланс между исследованием пространства решений и сохранением лучших особей.
5. Анализ параметров показал, что увеличение размера популяции и числа поколений улучшает качество решения, а вероятность мутации ~0.2 является хорошим компромиссом.
6. Полученное решение сравнено с известным оптимальным маршрутом длиной 538. Генетический алгоритм с CX-кроссинговером находит приемлемое решение, хотя для достижения глобального оптимума могут потребоваться дополнительные эвристики (например, 2-opt локальный поиск).